# ENG2040 assignment 1 - marketplace orders to a model-ready dataset

whole pipeline in one notebook, one section per step. `uv run python run.py` checks the supplied files, rebuilds `data/marketplace.sqlite` from `supplied/data/raw.sql` and runs this top to bottom.

each step writes its outputs to `data/<step>/` plus a `resources.json`. memory is tracked per step by sampling this kernel's rss every 10 ms while the step runs. rss at the start of the step is saved too, since the kernel still holds data from earlier steps.

In [1]:
import json, resource, sqlite3, subprocess, sys, threading, time
from pathlib import Path
import adbc_driver_sqlite.dbapi as adbc
import pandas as pd
import psutil

ROOT = Path.cwd()
DB = ROOT / "data" / "marketplace.sqlite"
con = adbc.connect(f"file:{DB}?mode=ro")  # read only, the db is never modified
q = lambda s: pd.read_sql(s, con, dtype_backend="pyarrow")  # adbc hands results over as arrow columns
proc = psutil.Process()
mb = lambda b: round(b / 1e6, 2)

def track(step):
    # call at the start of a step, call the returned function at the end
    out = ROOT / "data" / step
    out.mkdir(parents=True, exist_ok=True)
    stop, start, peak, t = threading.Event(), proc.memory_info().rss, [proc.memory_info().rss], time.perf_counter()
    def sample():
        while not stop.wait(0.01):
            peak[0] = max(peak[0], proc.memory_info().rss)
    th = threading.Thread(target=sample, daemon=True)
    th.start()
    def done(**extra):
        stop.set(); th.join()
        res = {"step": step, "seconds": round(time.perf_counter() - t, 2), "rss_start_MB": mb(start), "rss_peak_MB": mb(peak[0]),
               "rss_peak_above_start_MB": mb(peak[0] - start),
               "saved_MB": {f.name: mb(f.stat().st_size) for f in sorted(out.iterdir()) if f.name != "resources.json"}, **extra}
        (out / "resources.json").write_text(json.dumps(res, indent=1))
        return res
    return out, done

## step 1 - loading the data

one row per order from the supplied sqlite db. no cleaning in this step. only joining and aggregating, and every count and total gets checked against the db.

in: `data/marketplace.sqlite`

out, in `data/01_load/`:
- `orders.parquet` one row per order
- `items.parquet` one row per order item with its product and seller attributes (for the feature step)
- `checks.csv` every reconciliation check
- `load_comparison.csv` sql vs pandas runtime and memory
- `resources.json` runtime, memory, file sizes

In [2]:
OUT, done = track("01_load")

## tables and keys

- `orders` one row per order, key `order_id`. `customer_id` is the customer record for that one order
- `customers` key `customer_id`. `customer_unique_id` links the same person across orders
- `order_items` one row per purchased item, key `(order_id, order_item_id)`
- `order_payments` one row per payment record, key `(order_id, payment_sequential)`. `payment_value` is in centavos
- `order_reviews` key `(review_id, order_id)`. an order can have several reviews and one review can sit on several orders
- `products`, `sellers` lookups by `product_id`, `seller_id`
- `category_translation` portuguese category -> english

so orders is the grain. items, payments and reviews are one-to-many from an order so they get aggregated to one row per order first. joining them raw would multiply rows (3 items x 2 payments = 6 rows for one order) and double count totals.

the db was loaded with `foreign_keys=OFF` so nothing guarantees the keys actually match. checking that first:

In [3]:
tables = q("select name from sqlite_master where type='table' order by name").name
counts = pd.Series({t: q(f"select count(*) n from {t}").n[0] for t in tables})
print(counts)

links = q('''
select 'orders without a customer row' link, count(*) n from orders o left join customers c on c.customer_id = o.customer_id where c.customer_id is null
union all select 'customers without an order', count(*) from customers c left join orders o on o.customer_id = c.customer_id where o.order_id is null
union all select 'items without an order', count(*) from order_items i left join orders o on o.order_id = i.order_id where o.order_id is null
union all select 'items without a product row', count(*) from order_items i left join products p on p.product_id = i.product_id where p.product_id is null
union all select 'items without a seller row', count(*) from order_items i left join sellers s on s.seller_id = i.seller_id where s.seller_id is null
union all select 'payments without an order', count(*) from order_payments p left join orders o on o.order_id = p.order_id where o.order_id is null
union all select 'reviews without an order', count(*) from order_reviews r left join orders o on o.order_id = r.order_id where o.order_id is null
union all select 'orders with no items', count(*) from orders o where not exists (select 1 from order_items i where i.order_id = o.order_id)
union all select 'orders with no payments', count(*) from orders o where not exists (select 1 from order_payments p where p.order_id = o.order_id)
union all select 'orders with no reviews', count(*) from orders o where not exists (select 1 from order_reviews r where r.order_id = o.order_id)
union all select 'orders with 2+ reviews', count(*) from (select order_id from order_reviews group by order_id having count(*) > 1)
''')
links

category_translation       73
customers               20000
order_items             22662
order_payments          20875
order_reviews           19966
orders                  20000
products                11334
sellers                  2071
dtype: int64


,link,n
0,orders without a customer row,769
1,customers without an order,769
2,items without an order,0
3,items without a product row,910
4,items without a seller row,893
5,payments without an order,0
6,reviews without an order,0
7,orders with no items,142
8,orders with no payments,0
9,orders with no reviews,147


every item, payment and review points at a real order, so nothing gets lost when they roll up to orders. the other direction has gaps (orders with no customer row, no items, items whose product or seller row is missing). keys are all 32 char lowercase hex, so these are real missing rows and not formatting.

hence every join below is a left join from the order (or item) side. a missing customer or product just gives nulls and a flag, the order stays. dropping them is a cleaning decision for step 2, not something loading should do quietly.

the 769 customers with no order cant appear in a one-row-per-order table. they are counted here and left in the db.

## one row per order

aggregation happens inside sqlite. `order_items` and `order_payments` have `order_id` first in their primary key and `order_reviews` has an index on it, so each group by is one ordered pass over an index. only 20000 aggregated rows cross into python instead of ~117k raw ones. they cross through adbc, which hands them to pandas as arrow columns. pythons built in `sqlite3` driver makes one python object per value first, which turned out to be the real memory cost (see the comparison below).

sums ignore nulls in sql, so null counts are kept next to each sum. that way a total can always be traced back. `payment_value` stays in integer centavos here so the totals stay exact. converting to BRL happens in step 2.

In [4]:
ORDERS_SQL = '''
with it as (
  select order_id, count(*) n_items, count(distinct product_id) n_products, count(distinct seller_id) n_sellers,
         sum(price) price_sum, sum(freight_value) freight_sum,
         sum(price is null) n_price_null, sum(freight_value is null) n_freight_null
  from order_items group by order_id),
pay as (
  select order_id, count(*) n_payments, sum(payment_value) payment_sum_centavos, sum(payment_value is null) n_payment_null
  from order_payments group by order_id),
rev as (
  select order_id, count(*) n_reviews from order_reviews group by order_id)
select o.*,
       c.customer_id is not null has_customer, c.customer_unique_id, c.customer_zip_code_prefix, c.customer_city, c.customer_state,
       coalesce(it.n_items, 0) n_items, it.n_products, it.n_sellers, it.price_sum, it.freight_sum, it.n_price_null, it.n_freight_null,
       coalesce(pay.n_payments, 0) n_payments, pay.payment_sum_centavos, pay.n_payment_null,
       coalesce(rev.n_reviews, 0) n_reviews
from orders o
left join customers c on c.customer_id = o.customer_id
left join it on it.order_id = o.order_id
left join pay on pay.order_id = o.order_id
left join rev on rev.order_id = o.order_id
'''
orders = q(ORDERS_SQL)
orders["has_customer"] = orders.has_customer.astype("bool[pyarrow]")
print(orders.shape, orders.memory_usage(deep=True).sum() / 1e6, "MB in memory")
orders.head(3).T

(20000, 24) 7.017963 MB in memory


,0,1,2
order_id,47770eb9100c2d0c44946d9cf07ec65d,76c6e866289321a7c93b82b54852dc33,403b97836b0c04a622354cf531062e5f
customer_id,41ce2a54c0b03bf3443c3d931a367089,f54a9f0e6b351c431402b8461ea51999,7acc63bb81a1111a922c238549b85dde
order_status,delivered,delivered,delivered
order_purchase_timestamp,<NA>,2017-01-23 18:29:09,2018-01-02 19:00:43
order_approved_at,2018-08-08 08:55:23,2017-01-25 02:50:47,2018-01-02 19:09:04
order_delivered_carrier_date,2018-08-08 13:50:00,2017-01-26 14:16:31,2018-01-03 18:19:09
order_delivered_customer_date,2018-08-17 18:06:29,2017-02-02 14:08:10,2018-01-20 01:38:59
order_estimated_delivery_date,2018-09-04 00:00:00,2017-03-06 00:00:00,2018-02-06 00:00:00
has_customer,True,True,False
customer_unique_id,3a653a41f6f9fc3d2a113cf8398680e8,39382392765b6dc74812866ee5ee92a7,<NA>


## one row per item

the feature step needs product and seller attributes, which live at item level. same idea, left joins from the item side. category translation is left out on purpose. category names need normalising first (step 2) or the join silently misses.

In [5]:
ITEMS_SQL = '''
select i.*,
       p.product_id is not null has_product, p.product_category_name, p.product_name_lenght, p.product_description_lenght,
       p.product_photos_qty, p.product_weight_g, p.product_length_cm, p.product_height_cm, p.product_width_cm,
       s.seller_id is not null has_seller, s.seller_zip_code_prefix, s.seller_city, s.seller_state
from order_items i
left join products p on p.product_id = i.product_id
left join sellers s on s.seller_id = i.seller_id
'''
items = q(ITEMS_SQL)
items[["has_product", "has_seller"]] = items[["has_product", "has_seller"]].astype("bool[pyarrow]")
print(items.shape, items.memory_usage(deep=True).sum() / 1e6, "MB in memory")

(22662, 20) 5.868028 MB in memory


## checks: are records, counts and totals preserved

every number on the left comes from the raw tables, every number on the right from the output. totals are compared with a 1e-6 tolerance for the float sums. centavos are integers so they have to match exactly.

In [6]:
db = q('''select (select count(*) from orders), (select count(distinct order_id) from orders),
  (select count(*) from order_items), (select sum(price) from order_items), (select sum(freight_value) from order_items),
  (select count(price) from order_items), (select count(*) from order_payments), (select sum(payment_value) from order_payments),
  (select count(*) from order_reviews)''').iloc[0].tolist()
out = [len(orders), orders.order_id.nunique(),
       orders.n_items.sum(), orders.price_sum.sum(), orders.freight_sum.sum(),
       orders.n_items.sum() - orders.n_price_null.sum(), orders.n_payments.sum(), orders.payment_sum_centavos.sum(),
       orders.n_reviews.sum()]
names = ["orders rows", "unique order_id", "item rows", "price total BRL", "freight total BRL",
         "non-null prices", "payment rows", "payment total centavos", "review rows"]
checks = pd.DataFrame({"check": names, "db": db, "output": out})
checks.loc[len(checks)] = ["items.parquet rows", db[2], len(items)]
checks.loc[len(checks)] = ["items.parquet price total BRL", db[3], items.price.sum()]
checks["ok"] = [abs(float(a) - float(b)) <= 1e-6 * max(1, abs(float(a))) for a, b in zip(checks.db, checks.output)]
assert checks.ok.all()
checks

,check,db,output,ok
0,orders rows,2.000000e+04,2.000000e+04,True
1,unique order_id,2.000000e+04,2.000000e+04,True
2,item rows,2.266200e+04,2.266200e+04,True
3,price total BRL,5.444621e+07,5.444621e+07,True
4,freight total BRL,1.051506e+07,1.051506e+07,True
5,non-null prices,2.222300e+04,2.222300e+04,True
6,payment rows,2.087500e+04,2.087500e+04,True
7,payment total centavos,2.805245e+08,2.805245e+08,True
8,review rows,1.996600e+04,1.996600e+04,True
9,items.parquet rows,2.266200e+04,2.266200e+04,True


In [7]:
orders.to_parquet(OUT / "orders.parquet", index=False)
items.to_parquet(OUT / "items.parquet", index=False)
checks.to_csv(OUT / "checks.csv", index=False)
links.to_csv(OUT / "key_links.csv", index=False)
# reading back gives the same frames, so parquet keeps the types and values
assert pd.read_parquet(OUT / "orders.parquet", dtype_backend="pyarrow").equals(orders)
assert pd.read_parquet(OUT / "items.parquet", dtype_backend="pyarrow").equals(items)
done(in_memory_MB={"orders": mb(orders.memory_usage(deep=True).sum()), "items": mb(items.memory_usage(deep=True).sum())})

{'step': '01_load',
 'seconds': 0.31,
 'rss_start_MB': 130.86,
 'rss_peak_MB': 215.55,
 'rss_peak_above_start_MB': 84.69,
 'saved_MB': {'checks.csv': 0.0,
  'items.parquet': 1.86,
  'key_links.csv': 0.0,
  'load_comparison.csv': 0.0,
  'load_comparison_raw.csv': 0.0,
  'orders.parquet': 3.29},
 'in_memory_MB': {'orders': np.float64(7.02), 'items': np.float64(5.87)}}

## why this way: runtime, memory, storage

this runs after `done()` so it doesnt count toward step 1 resources. the alternative is reading every table whole into pandas and doing the same group by and merge there (same output columns). each is run through two drivers: pythons built in `sqlite3` and `adbc`.

the claim to test: pandas holds every raw row of every table at once, so its memory grows with the raw data. the sql version only hands python the aggregated rows. to test it the extract is copied 1x, 3x and 10x with new ids (`_0`, `_1`, ...), keeping the original schema and indexes. each approach runs 3 times per size in its own fresh process. a baseline run per driver only imports pandas and opens the db, so the cost of python + pandas on their own is subtracted out.

estimate written before the first run (sqlite3 driver only): at 1x both under a second and a few MB apart, both peaks dominated by the ~90 MB baseline. at 10x pandas should need clearly more memory above baseline than sql, maybe 2x.

In [8]:
TABLES = ["orders", "customers", "order_items", "order_payments", "order_reviews"]

def scaled_db(k):
    path = OUT / f"_scale{k}.sqlite"
    path.unlink(missing_ok=True)
    s = sqlite3.connect(path)
    s.execute(f"attach '{DB}' as src")
    for (sql,) in s.execute(f"select sql from src.sqlite_master where tbl_name in {tuple(TABLES)} and sql is not null order by type desc").fetchall():
        s.execute(sql)
    for t in TABLES:
        cols = [r[1] for r in s.execute(f"pragma src.table_info({t})")]
        for c in range(k):
            sel = ", ".join(f"{x} || '_{c}'" if x in ("order_id", "customer_id") else x for x in cols)
            s.execute(f"insert into {t} select {sel} from src.{t}")
    s.commit(); s.close()
    return path

CONNECT = {
    "sqlite3": "import sqlite3\ncon = sqlite3.connect('file:' + sys.argv[1] + '?mode=ro', uri=True)\n",
    "adbc": "import adbc_driver_sqlite.dbapi as adbc\ncon = adbc.connect('file:' + sys.argv[1] + '?mode=ro')\n",
}
SETUP = "import resource, sys, time\nimport pandas as pd\n"
REPORT = """
peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (1e6 if sys.platform == "darwin" else 1e3)
print(time.perf_counter() - t, peak)
"""
PANDAS = """
r = lambda t: pd.read_sql(f"select * from {t}", con, dtype_backend="pyarrow")
o, c, i, p, rv = r("orders"), r("customers"), r("order_items"), r("order_payments"), r("order_reviews")
i["price_null"], i["freight_null"], p["payment_null"] = i.price.isna(), i.freight_value.isna(), p.payment_value.isna()
it = i.groupby("order_id").agg(n_items=("order_item_id", "size"), n_products=("product_id", "nunique"), n_sellers=("seller_id", "nunique"),
    price_sum=("price", "sum"), freight_sum=("freight_value", "sum"), n_price_null=("price_null", "sum"), n_freight_null=("freight_null", "sum")).reset_index()
pay = p.groupby("order_id").agg(n_payments=("payment_sequential", "size"), payment_sum_centavos=("payment_value", "sum"), n_payment_null=("payment_null", "sum")).reset_index()
rev = rv.groupby("order_id").size().rename("n_reviews").reset_index()
df = o.merge(c, on="customer_id", how="left").merge(it, on="order_id", how="left").merge(pay, on="order_id", how="left").merge(rev, on="order_id", how="left")
"""
SQL = f"df = pd.read_sql({ORDERS_SQL!r}, con, dtype_backend='pyarrow')\n"
runs = {"baseline": "", "sql": SQL, "pandas": PANDAS}
rows = []
for k in [1, 3, 10]:
    path = scaled_db(k)
    for driver, connect in CONNECT.items():
        for name, body in runs.items():
            for rep in range(3):
                code = SETUP + connect + "t = time.perf_counter()\n" + body + REPORT
                s, peak = subprocess.run([sys.executable, "-c", code, str(path)], capture_output=True, text=True, check=True).stdout.split()
                rows.append((k, driver, name, rep, float(s), float(peak), mb(path.stat().st_size)))
    path.unlink()  # temporary, deleted straight away
raw = pd.DataFrame(rows, columns=["scale", "driver", "approach", "rep", "seconds", "peak_MB", "db_MB"])
med = raw.groupby(["scale", "driver", "approach"], sort=False)[["seconds", "peak_MB", "db_MB"]].median()
base = med.xs("baseline", level="approach").peak_MB
cmp = med.drop("baseline", level="approach").reset_index()
cmp["above_baseline_MB"] = cmp.peak_MB - base.loc[list(zip(cmp.scale, cmp.driver))].values
cmp = cmp.pivot_table(index="scale", columns=["driver", "approach"], values=["above_baseline_MB", "seconds"]).round(2)
raw.to_csv(OUT / "load_comparison_raw.csv", index=False)
cmp.to_csv(OUT / "load_comparison.csv")
cmp

above_baseline_MB                         seconds                    
driver                adbc         sqlite3            adbc       sqlite3      
approach            pandas     sql  pandas     sql  pandas   sql  pandas   sql
scale                                                                         
1                    52.64   24.77   56.46   51.12    0.08  0.10    0.14  0.26
3                   134.53   50.66  131.01  131.35    0.23  0.44    0.41  0.97
10                  421.27  120.00  382.89  390.77    0.74  1.84    1.44  3.72

measured. the first version only had the `sqlite3` driver, and there the estimate was wrong. sql and pandas grew at the same rate (both ~400 MB above baseline at 10x) and pandas was faster at every size. so "sql scales better" was not true as built.

the cost was not the sql. it was the driver. `sqlite3` turns every value of the result into its own python object before pandas packs it into columns (20000 order rows were ~30 MB as python objects vs ~8 MB as a dataframe). with `adbc` the same sql result goes straight into arrow columns.

with adbc for both: sql needs a few times less memory above baseline than pandas, and the gap widens with size since pandas holds every raw row at once. pandas is still faster, a bit over 2x at 10x, but both stay within a couple of seconds at 10x.

so the choice is sql aggregation through adbc. memory is what decides whether a step fits on a machine at all, while a second or two of runtime doesnt matter for a batch step like this. if runtime mattered more than memory, pandas through adbc would be the pick.

storage. same orders table as csv vs parquet:

In [9]:
tmp = OUT / "_orders_tmp.csv"
orders.to_csv(tmp, index=False)
storage = pd.Series({"csv": tmp.stat().st_size / 1e6, "parquet": (OUT / "orders.parquet").stat().st_size / 1e6}).round(2)
tmp.unlink()
print(storage, "MB")

csv        5.25
parquet    3.29
dtype: float64 MB


## step 2 - data quality and cleaning

in: `data/01_load/orders.parquet`, `items.parquet`, plus payments and the category translation from the db

out, in `data/02_clean/`:
- `orders_clean.parquet` one row per order, all 20000 kept. cleaned fields, `delivered_after_estimate` label (or missing) and `unlabelled_reason`
- `items_clean.parquet` one row per order item, cleaned, with english category
- `changes.parquet` one row per changed value: table, key, column, rule, old, new
- `cleaning_summary.csv` the same changes counted per rule
- `record_accounting.csv` where every order ended up
- `payment_reconciliation.csv` how many orders match their payments before and after cleaning

nothing is dropped. an order that cant get a reliable label keeps its row and gets a reason instead. fixes are only applied where there is evidence for them (see the payment check), everything else is set to missing or flagged.

In [10]:
OUT, done = track("02_clean")
o = pd.read_parquet(ROOT / "data" / "01_load" / "orders.parquet", dtype_backend="pyarrow")
it = pd.read_parquet(ROOT / "data" / "01_load" / "items.parquet", dtype_backend="pyarrow")
pay = q("select * from order_payments")
tr = q("select * from category_translation")
it["item_key"] = it.order_id + "/" + it.order_item_id.astype("string[pyarrow]")
log = []

def change(table, df, key, col, mask, new, rule):
    # apply new values where mask is true and keep a record of every old -> new
    m = mask.fillna(False).astype(bool)
    if not m.any():
        return
    new = new if isinstance(new, pd.Series) else pd.Series(new, index=df.index, dtype=df[col].dtype)
    log.append(pd.DataFrame({"table": table, "key": df.loc[m, key].astype(str), "column": col, "rule": rule,
                             "old": df.loc[m, col].astype(str), "new": new[m].astype(str)}))
    df.loc[m, col] = new[m]

### what is wrong, found by profiling

counted on the step 1 output before changing anything:

In [11]:
TS = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]
UF = {"AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"}
parse = lambda s: pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S", errors="coerce")
d = {c: parse(o[c]) for c in TS}
st = o.order_status.str.strip().str.lower()
profile = pd.Series({
    "status with extra spaces or capitals": (o.order_status != st).sum(),
    "status not_reported": (st == "not_reported").sum(),
    "orders with an impossible date (2018-02-30)": o[TS].eq("2018-02-30 10:00:00").any(axis=1).sum(),
    "delivered before approval": (d["order_delivered_customer_date"] < d["order_approved_at"]).sum(),
    "  of which exactly 7 days before": ((d["order_approved_at"] - d["order_delivered_customer_date"]) == pd.Timedelta(days=7)).sum(),
    "carrier pickup before approval": (d["order_delivered_carrier_date"] < d["order_approved_at"]).sum(),
    "estimate before approval": (d["order_estimated_delivery_date"] < d["order_approved_at"]).sum(),
    "cancelled but has a delivery date": ((st == "canceled") & d["order_delivered_customer_date"].notna()).sum(),
    "delivered but no delivery date": ((st == "delivered") & o.order_delivered_customer_date.isna()).sum(),
    "customer state not a brazilian state": (~o.customer_state.str.strip().str.upper().isin(UF) & o.customer_state.notna()).sum(),
    "item price negative": (it.price < 0).sum(),
    "item price > 5000": (it.price > 5000).sum(),
    "item freight negative": (it.freight_value < 0).sum(),
    "item freight > 500": (it.freight_value > 500).sum(),
    "item price missing": it.price.isna().sum(),
    "item freight missing": it.freight_value.isna().sum(),
    "item category unmapped_category": (it.product_category_name == "unmapped_category").sum(),
    "item size or weight <= 0": (it[["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]] <= 0).any(axis=1).sum(),
    "payment negative": (pay.payment_value < 0).sum(),
    "payment missing": pay.payment_value.isna().sum(),
    "payment type unclassified_payment": (pay.payment_type == "unclassified_payment").sum(),
})
profile

status with extra spaces or capitals           783
status not_reported                            804
orders with an impossible date (2018-02-30)    797
delivered before approval                      812
  of which exactly 7 days before               803
carrier pickup before approval                 270
estimate before approval                         2
cancelled but has a delivery date              733
delivered but no delivery date                 180
customer state not a brazilian state           838
item price negative                            408
item price > 5000                              417
item freight negative                          458
item freight > 500                             473
item price missing                             439
item freight missing                           427
item category unmapped_category                674
item size or weight <= 0                       632
payment negative                               975
payment missing                

### is a fix supported by evidence? check against payments

price and freight feed the features, so a wrong fix would leak straight into the model. payments are an independent record of what the customer paid, so for orders with exactly one kind of problem the check is: after the fix, does price + freight equal the payment (to the centavo)?

- negative values -> sign flip, so take the absolute value
- values > 5000 (price) or > 500 (freight) -> try both /100 and /1000

the thresholds sit in clear gaps: normal prices top out at 4690 and the big ones start at 6499, normal freight tops out at 314 and the big ones start at 1000.

In [12]:
paid = (o.set_index("order_id").payment_sum_centavos / 100)
complete = o.set_index("order_id").eval("n_price_null == 0 and n_freight_null == 0 and n_payment_null == 0 and n_items > 0")
kind = pd.DataFrame({"order_id": it.order_id, "price neg": it.price < 0, "price big": it.price > 5000,
                     "freight neg": it.freight_value < 0, "freight big": it.freight_value > 500}).fillna(False).groupby("order_id").any()
single = kind.sum(axis=1) == 1

def match_rate(p, f, ids):
    t = pd.DataFrame({"order_id": it.order_id, "t": p + f}).groupby("order_id").t.sum()
    return round(float(((paid[ids] - t[ids]).abs() <= 0.01).mean()), 3)

big = lambda x, cap, k: x.where(~(x > cap).fillna(False), x / k)
rows = []
for k in kind:
    ids = kind.index[kind[k] & single].intersection(complete.index[complete])
    p, f = it.price, it.freight_value
    fixes = {"none": (p, f), "abs": (p.abs(), f.abs()), "/100": (big(p, 5000, 100), big(f, 500, 100)), "/1000": (big(p, 5000, 1000), big(f, 500, 1000))}
    rows.append({"problem": k, "orders": len(ids), **{fx: match_rate(*v, ids) for fx, v in fixes.items()}})
evidence = pd.DataFrame(rows).set_index("problem")
evidence

,orders,none,abs,/100,/1000
problem,,,,,
price neg,366,0.000,0.872,0.0,0.000
price big,384,0.003,0.003,0.0,0.935
freight neg,413,0.000,0.903,0.0,0.000
freight big,441,0.000,0.000,0.0,0.884


so abs for negatives and /1000 for the big values. /100 matches nothing, so the obvious guess (centavos typed in as reais) is wrong, the values are 1000x. orders that still dont match after the fix mostly have some other problem in their payments (negative or missing payment, see below).

### apply the cleaning rules

In [13]:
# orders
change("orders", o, "order_id", "order_status", o.order_status != st, st, "status: strip spaces + lowercase")
for c in TS:
    change("orders", o, "order_id", c, o[c].notna() & parse(o[c]).isna(), None, "impossible calendar date -> missing")
    o[c] = parse(o[c])
cs = o.customer_state.str.strip().str.upper()
change("orders", o, "order_id", "customer_state", o.customer_state != cs, cs, "state: strip spaces + uppercase")
change("orders", o, "order_id", "customer_state", ~o.customer_state.isin(UF) & o.customer_state.notna(), None, "state not a brazilian state (ZZ or blank) -> missing")

# items
for c, cap in [("price", 5000), ("freight_value", 500)]:
    change("items", it, "item_key", c, it[c] < 0, it[c].abs(), "negative -> absolute value (sign flip, checked against payments)")
    change("items", it, "item_key", c, it[c] > cap, it[c] / 1000, "1000x entry error -> /1000 (checked against payments)")
cat = it.product_category_name.str.strip().str.lower()
change("items", it, "item_key", "product_category_name", it.product_category_name != cat, cat, "category: strip spaces + lowercase")
change("items", it, "item_key", "product_category_name", it.product_category_name == "unmapped_category", None, "placeholder category -> missing")
for c in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    change("items", it, "item_key", c, it[c] <= 0, None, "size or weight <= 0 -> missing")
ss = it.seller_state.str.strip().str.upper()
change("items", it, "item_key", "seller_state", it.seller_state != ss, ss, "state: strip spaces + uppercase")
change("items", it, "item_key", "seller_state", ~it.seller_state.isin(UF) & it.seller_state.notna(), None, "state not a brazilian state (ZZ or blank) -> missing")

# a zip prefix belongs to one state, so a missing state can come back from the zip if that zip is seen elsewhere with exactly one state
zs = pd.concat([o[["customer_zip_code_prefix", "customer_state"]].set_axis(["zip", "state"], axis=1),
                it[["seller_zip_code_prefix", "seller_state"]].set_axis(["zip", "state"], axis=1)]).dropna()
n = zs.groupby("zip").state.nunique()
print("zip prefixes seen with a valid state:", len(n), "| seen with more than one state:", int((n > 1).sum()))
zmap = zs[zs.zip.isin(n.index[n == 1])].drop_duplicates("zip").set_index("zip").state
o["customer_state_recovered"] = (o.customer_state.isna() & o.customer_zip_code_prefix.isin(zmap.index)).fillna(False).astype("bool[pyarrow]")
change("orders", o, "order_id", "customer_state", o.customer_state_recovered, o.customer_zip_code_prefix.map(zmap), "missing state -> recovered from zip prefix")
change("items", it, "item_key", "seller_state", it.seller_state.isna() & it.seller_zip_code_prefix.isin(zmap.index), it.seller_zip_code_prefix.map(zmap), "missing state -> recovered from zip prefix")
print("customer state still missing:", o.customer_state.isna().sum(), "| seller state still missing:", it.seller_state.isna().sum())

tr["product_category_name"] = tr.product_category_name.str.strip().str.lower()
print("translation keys unique after normalising:", tr.product_category_name.is_unique)
it = it.merge(tr, on="product_category_name", how="left")
print("categories with no english name:", it.product_category_name[it.product_category_name_english.isna()].dropna().unique())

changes = pd.concat(log, ignore_index=True)
summary = changes.groupby(["table", "column", "rule"], sort=False).size().rename("values_changed").reset_index()
summary

zip prefixes seen with a valid state: 8486 | seen with more than one state: 14
customer state still missing: 1007 | seller state still missing: 1154
translation keys unique after normalising: True
categories with no english name: <ArrowExtensionArray>
['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
Length: 2, dtype: string[pyarrow]


,table,column,rule,values_changed
0,orders,order_status,status: strip spaces + lowercase,783
1,orders,order_purchase_timestamp,impossible calendar date -> missing,165
2,orders,order_approved_at,impossible calendar date -> missing,152
3,orders,order_delivered_carrier_date,impossible calendar date -> missing,149
4,orders,order_delivered_customer_date,impossible calendar date -> missing,161
5,orders,order_estimated_delivery_date,impossible calendar date -> missing,170
6,orders,customer_state,state: strip spaces + uppercase,877
7,orders,customer_state,state not a brazilian state (ZZ or blank) -> m...,838
8,items,price,"negative -> absolute value (sign flip, checked...",408
9,items,price,1000x entry error -> /1000 (checked against pa...,417


### labels

`delivered_after_estimate` = 1 if the delivery calendar date is after the estimated calendar date, 0 if on or before it. the recorded local clock is used as is, no timezone.

an order only gets a label if its outcome is a usable completed delivery. the first rule that applies gives the reason it stays unlabelled:

In [14]:
dl, ap, pu, es = o.order_delivered_customer_date, o.order_approved_at, o.order_purchase_timestamp, o.order_estimated_delivery_date
s = o.order_status
rules = [
    (s.isin(["canceled", "unavailable"]), "cancelled or unavailable"),
    (s.isin(["shipped", "invoiced", "processing", "created", "approved"]), "not delivered yet"),
    (s == "not_reported", "status unknown"),
    (o.n_items == 0, "delivered but has no items"),
    (dl.isna(), "delivery date missing or impossible"),
    ((dl < ap) | (dl < pu), "delivery dated before purchase or approval"),
    (es.isna(), "estimate missing or impossible"),
    (es < ap, "estimate dated before approval"),
]
reason = pd.Series(pd.NA, index=o.index, dtype="string[pyarrow]")
for m, r in rules:
    reason = reason.mask(reason.isna() & m.fillna(False).astype(bool), r)
o["unlabelled_reason"] = reason
o["days_late"] = (dl.dt.normalize() - es.dt.normalize()).dt.days.where(reason.isna())
o["delivered_after_estimate"] = (o.days_late > 0).astype("Int8").where(reason.isna())

accounting = o.unlabelled_reason.fillna("labelled").value_counts().rename("orders").reset_index()
assert accounting.orders.sum() == len(o) == 20000
print(o.delivered_after_estimate.value_counts(dropna=False))
accounting

delivered_after_estimate
0       15375
<NA>     3518
1        1107
Name: count, dtype: Int64


,unlabelled_reason,orders
0,labelled,16482
1,cancelled or unavailable,944
2,status unknown,804
3,delivery dated before purchase or approval,791
4,delivery date missing or impossible,333
5,not delivered yet,322
6,estimate missing or impossible,319
7,delivered but has no items,5


### recompute order totals from the cleaned items, then check against payments again

In [15]:
tot = it.groupby("order_id").agg(price_sum=("price", "sum"), freight_sum=("freight_value", "sum"))
before = (paid - (o.set_index("order_id").price_sum + o.set_index("order_id").freight_sum)).abs() <= 0.01
o = o.drop(columns=["price_sum", "freight_sum"]).merge(tot, on="order_id", how="left")
after = (paid - (o.set_index("order_id").price_sum + o.set_index("order_id").freight_sum)).abs() <= 0.01
gap = (paid - o.set_index("order_id").eval("price_sum + freight_sum"))[complete & ~after]
neg_pay = pay.groupby("order_id").payment_value.min().lt(0)
recon = pd.Series({
    "complete orders": complete.sum(),
    "match payments before cleaning": before[complete].sum(),
    "match payments after cleaning": after[complete].sum(),
    "still not matching": (~after[complete]).sum(),
    "  of which have a negative payment": neg_pay.reindex(gap.index, fill_value=False).sum(),
    "  of which paid exactly 7.23 more": (gap - 7.23).abs().le(0.011).sum(),
})
recon

complete orders                       18122
match payments before cleaning        14845
match payments after cleaning         16302
still not matching                     1820
  of which have a negative payment      919
  of which paid exactly 7.23 more       832
dtype: int64

### what stays unresolved and how it affects things

- payments: negative values, `unclassified_payment` types and -1 installments are left as they are. payments cant be predictors and arent needed for the label, they are only used above as an independent check
- 764 orders paid exactly 7.23 more than price + freight. it is not clear which side is wrong, so nothing is changed. if prices were the wrong side, those orders basket values are slightly off
- carrier pickup dated before approval is flagged in the profile but not changed. the carrier date isnt used for labels or features
- `not_reported` orders mostly have delivery dates, but with the status unknown the outcome isnt reliable, so they stay unlabelled. this is 804 orders (4%) taken out of every rate, so the analysis checks how the late rate moves if they are put back
- the 7 day shifted delivery dates could be shifted back, but that would be a guess about how the error happened, so those orders stay unlabelled
- states that couldnt be recovered from the zip stay missing (mostly orders with no customer row at all). recovered states are flagged in `customer_state_recovered` so step 3 can check results with and without them
- `pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos` have no row in the translation table, so their english name is missing. the portuguese name is still there
- missing prices, freight, category and sizes stay missing. a missing optional catalogue field doesnt make an order unusable

In [16]:
o.to_parquet(OUT / "orders_clean.parquet", index=False)
it.to_parquet(OUT / "items_clean.parquet", index=False)
changes.to_parquet(OUT / "changes.parquet", index=False)
summary.to_csv(OUT / "cleaning_summary.csv", index=False)
accounting.to_csv(OUT / "record_accounting.csv", index=False)
recon.rename("orders").to_csv(OUT / "payment_reconciliation.csv")
evidence.to_csv(OUT / "fix_evidence.csv")
profile.rename("count").to_csv(OUT / "profile.csv")
assert len(o) == 20000 and o.order_id.is_unique and len(it) == 22662
done(in_memory_MB={"orders": mb(o.memory_usage(deep=True).sum()), "items": mb(it.memory_usage(deep=True).sum()), "changes": mb(changes.memory_usage(deep=True).sum())})

{'step': '02_clean',
 'seconds': 0.29,
 'rss_start_MB': 138.26,
 'rss_peak_MB': 242.66,
 'rss_peak_above_start_MB': 104.4,
 'saved_MB': {'changes.parquet': 0.86,
  'cleaning_summary.csv': 0.0,
  'fix_evidence.csv': 0.0,
  'items_clean.parquet': 2.54,
  'orders_clean.parquet': 3.14,
  'payment_reconciliation.csv': 0.0,
  'profile.csv': 0.0,
  'record_accounting.csv': 0.0},
 'in_memory_MB': {'orders': np.float64(6.0),
  'items': np.float64(7.16),
  'changes': np.float64(4.45)}}